# ADVI for Taxi Trajectory Clustering
## Reproducing Section 4.4 of Kucukelbir et al. (2016)

**Reference:** *Automatic Differentiation Variational Inference*, Kucukelbir, A., Tran, D., Ranganath, R., Gelman, A., & Blei, D. M. (2016). arXiv:1603.00788.

---

### Objectives

This notebook implements the case study from Section 4.4 of the ADVI paper: **Exploring Millions of Taxi Trajectories** from the city of Porto.

The pipeline follows these steps:

1. **Data loading & preprocessing** — Load the Porto taxi dataset (1.7M rides, year 2014). Each trajectory is a variable-length sequence of (longitude, latitude) GPS coordinates recorded at 15-second intervals. We interpolate all trajectories to a fixed length of 50 coordinate pairs, yielding points in $\mathbb{R}^{100}$.

2. **Dimensionality reduction via PPCA with ARD** — Fit a Probabilistic PCA model with Automatic Relevance Determination on a subsample of 10 000 trajectories. PPCA-ARD automatically selects the effective number of principal components (the paper finds 11). Inference is performed using mean-field ADVI.

3. **Clustering via Gaussian Mixture Model** — Project all 1.7M trajectories into the learned subspace, then fit a GMM with $K=30$ components using ADVI with minibatch stochastic optimization.

4. **Visualization** — Display 50 000 randomly sampled trajectories colored by their cluster assignment on a map of Porto.

5. **Extension: Supervised PPCA (SUP-PPCA)** — Incorporate trip duration as a supervised signal into the dimensionality reduction step, then re-cluster. This reveals structure related to traffic patterns (e.g., inner vs. outer bridge routes).

### ADVI 

For each model above, we use ADVI to approximate the posterior $p(\theta \mid x)$ with a Gaussian $q(\zeta; \phi) = \mathcal{N}(\zeta; \mu, \Sigma)$ in a transformed unconstrained space $\zeta = T(\theta)$. The algorithm:
- Transforms constrained latent variables to $\mathbb{R}^K$
- Uses the reparameterization trick (elliptical standardization) to express ELBO gradients as expectations over $\mathcal{N}(0, I)$
- Approximates gradients via Monte Carlo (single sample suffices)
- Optimizes with adaptive stochastic gradient ascent

## 1. Imports

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.interpolate import interp1d

torch.set_default_dtype(torch.float64)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"NumPy {np.__version__} | PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NumPy 2.4.3 | PyTorch 2.10.0
CUDA available: False


## 2. Load the Porto Taxi Trajectories dataset

The dataset comes from the ECML-PKDD 2015 competition and contains **all 1.7 million taxi rides** in the city of Porto during the year 2014.

Each ride is described by:
- **POLYLINE**: a JSON-encoded list of `[longitude, latitude]` GPS coordinates sampled every 15 seconds.
- **TIMESTAMP**: Unix timestamp of the trip start.
- **TAXI_ID**: identifier of the taxi (442 taxis total).


In [2]:
import kagglehub

path = kagglehub.dataset_download("crailtap/taxi-trajectory")
print(f"Dataset downloaded to: {path}")

100%|██████████| 515M/515M [03:53<00:00, 2.31MB/s] 

Extracting files...


Dataset downloaded to: /Users/calla/.cache/kagglehub/datasets/crailtap/taxi-trajectory/versions/1


In [3]:
import os

csv_candidates = [
    os.path.join(path, "train.csv"),
    os.path.join(path, "train.csv.zip"),
]
csv_path = next((p for p in csv_candidates if os.path.exists(p)), None)
if csv_path is None:
    for root, dirs, files in os.walk(path):
        for f in files:
            if f.endswith(".csv") or f.endswith(".csv.zip"):
                csv_path = os.path.join(root, f)
                break

print(f"Loading from: {csv_path}")
df = pd.read_csv(csv_path)
print(f"Shape: {df.shape}")
df.head()

Loading from: /Users/calla/.cache/kagglehub/datasets/crailtap/taxi-trajectory/versions/1/train.csv
Shape: (1710670, 9)


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
0,1372636858620000589,C,NaN,NaN,20000589,1372636858,A,False,"[[-8.618643,41.141412],[-8.618499,41.141376],[..."
1,1372637303620000596,B,NaN,7.0,20000596,1372637303,A,False,"[[-8.639847,41.159826],[-8.640351,41.159871],[..."
2,1372636951620000320,C,NaN,NaN,20000320,1372636951,A,False,"[[-8.612964,41.140359],[-8.613378,41.14035],[-..."
3,1372636854620000520,C,NaN,NaN,20000520,1372636854,A,False,"[[-8.574678,41.151951],[-8.574705,41.151942],[..."
4,1372637091620000337,C,NaN,NaN,20000337,1372637091,A,False,"[[-8.645994,41.18049],[-8.645949,41.180517],[-..."


In [4]:
df.info()
print(f"\nNumber of unique taxis: {df['TAXI_ID'].nunique()}")
print(f"Columns: {list(df.columns)}")

<class 'pandas.DataFrame'>
RangeIndex: 1710670 entries, 0 to 1710669
Data columns (total 9 columns):
 #   Column        Dtype  
---  ------        -----  
 0   TRIP_ID       int64  
 1   CALL_TYPE     str    
 2   ORIGIN_CALL   float64
 3   ORIGIN_STAND  float64
 4   TAXI_ID       int64  
 5   TIMESTAMP     int64  
 6   DAY_TYPE      str    
 7   MISSING_DATA  bool   
 8   POLYLINE      str    
dtypes: bool(1), float64(2), int64(3), str(3)
memory usage: 106.0 MB

Number of unique taxis: 448
Columns: ['TRIP_ID', 'CALL_TYPE', 'ORIGIN_CALL', 'ORIGIN_STAND', 'TAXI_ID', 'TIMESTAMP', 'DAY_TYPE', 'MISSING_DATA', 'POLYLINE']


### 2.1 Parse polylines and compute trip durations

Each `POLYLINE` is a JSON list of `[lon, lat]` pairs sampled every 15 s. The trip duration (in seconds) is `15 * (len(polyline) - 1)`. We discard trips flagged as having missing data or with fewer than 2 GPS points.

In [5]:
df = df[df["MISSING_DATA"] == False].copy()

polylines = df["POLYLINE"].apply(json.loads)
lengths = polylines.apply(len)

# Drop trips with fewer than 2 GPS points (no trajectory to interpolate)
valid = lengths >= 2
polylines = polylines[valid]
df = df[valid].copy()
durations = (polylines.apply(len) - 1) * 15  # seconds

df["polyline"] = polylines
df["duration_s"] = durations.values
df["n_points"] = polylines.apply(len)

print(f"Valid trajectories: {len(df):,}")
print(f"Average trip duration: {df['duration_s'].mean():.0f} s  ({df['duration_s'].mean()/60:.1f} min)")
print(f"Average GPS points per trip: {df['n_points'].mean():.1f}")
print(f"Median GPS points per trip: {df['n_points'].median():.1f}")

Valid trajectories: 1,674,152
Average trip duration: 732 s  (12.2 min)
Average GPS points per trip: 49.8
Median GPS points per trip: 42.0


### 2.2 Interpolate trajectories to a fixed length

Following the paper: *"The average trip is approximately 13 minutes long, which corresponds to 50 coordinates. We want to cluster independent of length, so we interpolate all trajectories to 50 coordinate pairs."*

This converts each trajectory into a point in $\mathbb{R}^{100}$ (50 longitudes + 50 latitudes, interleaved as `[lon_1, lat_1, ..., lon_50, lat_50]`).

In [ ]:
N_INTERP = 50  # fixed number of coordinate pairs per trajectory

def interpolate_trajectory(coords, n_points=N_INTERP):
    """Linearly interpolate a trajectory to exactly n_points (lon, lat) pairs."""
    coords = np.array(coords)
    n = len(coords)
    t_original = np.linspace(0, 1, n)
    t_new = np.linspace(0, 1, n_points)
    lon_interp = interp1d(t_original, coords[:, 0], kind="linear")(t_new)
    lat_interp = interp1d(t_original, coords[:, 1], kind="linear")(t_new)
    # Interleave: [lon1, lat1, lon2, lat2, ..., lon50, lat50]
    return np.column_stack([lon_interp, lat_interp]).ravel()

print(f"Interpolating {len(df):,} trajectories to {N_INTERP} coordinate pairs...")
X = np.vstack(df["polyline"].apply(interpolate_trajectory).values)
durations = df["duration_s"].values

print(f"Trajectory matrix X: shape {X.shape}  (N trajectories x {2*N_INTERP} features)")
print(f"Duration vector:     shape {durations.shape}")

Interpolating 1,674,152 trajectories to 50 coordinate pairs...


### 2.3 Quick sanity check

Plot a few random trajectories to make sure the data looks reasonable.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sample_idx = np.random.choice(len(X), size=3, replace=False)
for ax, idx in zip(axes, sample_idx):
    traj = X[idx].reshape(N_INTERP, 2)
    ax.plot(traj[:, 0], traj[:, 1], "o-", markersize=2, linewidth=0.8)
    ax.plot(traj[0, 0], traj[0, 1], "gs", markersize=8, label="start")
    ax.plot(traj[-1, 0], traj[-1, 1], "r^", markersize=8, label="end")
    ax.set_title(f"Trip {idx} — {durations[idx]/60:.1f} min")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.legend(fontsize=8)
    ax.set_aspect("equal")

plt.suptitle("Sample interpolated trajectories", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nDataset ready: {X.shape[0]:,} trajectories in R^{X.shape[1]}")
print(f"Duration range: {durations.min():.0f}s – {durations.max():.0f}s")

## 3. Dimensionality Reduction: PPCA with Automatic Relevance Determination

### 3.1 The PPCA-ARD Model (Appendix F.6)

Probabilistic PCA (PPCA) is a Bayesian version of classical PCA. It models each observation $x_n \in \mathbb{R}^D$ as a noisy linear projection of a low-dimensional latent code $z_n \in \mathbb{R}^M$ through a weight matrix $W \in \mathbb{R}^{D \times M}$.

**Automatic Relevance Determination (ARD)** extends PPCA with a hierarchical prior on $W$ that automatically prunes unnecessary latent dimensions. We set $M = D$ (the maximum) and let the model decide how many components to keep.

#### Generative model

$$z_n \sim \mathcal{N}(0, I_M) \quad \text{for } n = 1, \ldots, N$$

$$\alpha_m \sim \text{InvGamma}(1, 1) \quad \text{for } m = 1, \ldots, M$$

$$w_d \sim \mathcal{N}\!\left(0,\; \sigma \cdot \text{diag}(\alpha)\right) \quad \text{for } d = 1, \ldots, D$$

$$\sigma \sim \text{LogNormal}(0, 1)$$

$$x_n \sim \mathcal{N}(W z_n,\; \sigma^2 I_D) \quad \text{for } n = 1, \ldots, N$$

where $w_d \in \mathbb{R}^M$ is the $d$-th row of $W$.

#### How ARD works

Each $\alpha_m$ controls the prior variance of the $m$-th column of $W$:
- **Large $\alpha_m$** → wide prior → the model is free to use component $m$ → **active dimension**
- **Small $\alpha_m$** → tight prior → the $m$-th column of $W$ is shrunk toward zero → **pruned dimension**

After inference, we inspect the posterior means $\hat{\alpha}_1, \ldots, \hat{\alpha}_M$ and keep only the components where $\hat{\alpha}_m$ is large. The paper finds an **11-dimensional** subspace for the taxi data.

### 3.2 Mean-Field ADVI — the algorithm

We want to approximate the posterior $p(\theta \mid x)$ where $\theta = \{z_{1:N}, W, \sigma, \alpha\}$ collects **all** latent variables. Exact inference is intractable, so we use ADVI.

#### Step 1 — Transform to unconstrained space

Some latent variables are constrained:

| Variable | Constraint | Transform $T$ | Unconstrained $\zeta$ | Jacobian $\log\lvert\tfrac{d\theta}{d\zeta}\rvert$ |
|----------|-----------|---------------|----------------------|---------------------------------------------|
| $z_n, W$ | $\mathbb{R}$ (none) | identity | $\zeta = \theta$ | $0$ |
| $\sigma$ | $\mathbb{R}_{>0}$ | $\log$ | $\zeta_\sigma = \log \sigma$ | $\log \sigma$ |
| $\alpha_m$ | $\mathbb{R}_{>0}$ | $\log$ | $\zeta_{\alpha_m} = \log \alpha_m$ | $\log \alpha_m$ |

After this transform, all $K$ latent variables live in $\mathbb{R}^K$ with $K = MN + DM + 1 + M$.

#### Step 2 — Mean-field Gaussian approximation

We posit a fully factorized Gaussian in the unconstrained space:

$$q(\zeta;\, \phi) = \prod_{k=1}^{K} \mathcal{N}\!\left(\zeta_k;\; \mu_k,\, e^{2\omega_k}\right)$$

The variational parameters $\phi = (\mu, \omega) \in \mathbb{R}^{2K}$ are **unconstrained** ($\omega_k = \log \sigma_{q,k}$).

#### Step 3 — The ELBO

The Evidence Lower Bound in the transformed space is (Eq. 5 of the paper):

$$\mathcal{L}(\phi) = \underbrace{\mathbb{E}_{q(\zeta;\phi)}\!\left[\log p\!\left(x, T^{-1}(\zeta)\right) + \log\!\left|\det J_{T^{-1}}(\zeta)\right|\right]}_{\text{expected log-joint + Jacobian}} + \underbrace{H\!\left[q(\zeta;\phi)\right]}_{\text{entropy}}$$

The entropy of the mean-field Gaussian has a closed form:

$$H[q] = \sum_{k=1}^{K} \omega_k + \frac{K}{2}\left(1 + \log 2\pi\right)$$

#### Step 4 — Reparameterization trick (elliptical standardization)

To compute gradients of $\mathcal{L}$ w.r.t. $\phi$, we reparameterize:

$$\zeta = \mu + e^{\omega} \odot \eta, \quad \eta \sim \mathcal{N}(0, I_K)$$

This makes the expectation independent of $\phi$ — we sample $\eta$ from a **fixed** standard Gaussian. The gradient flows through $\zeta(\mu, \omega, \eta)$ into the log-joint via standard backpropagation (automatic differentiation).

#### Step 5 — Stochastic gradient ascent

At each iteration $i$:

1. Draw $\eta \sim \mathcal{N}(0, I_K)$ (one MC sample suffices, $M=1$)
2. Compute $\zeta = \mu + e^\omega \odot \eta$
3. Evaluate $\widehat{\mathcal{L}} = \log p(x, T^{-1}(\zeta)) + \log|\det J_{T^{-1}}(\zeta)| + H[q]$
4. Compute $\nabla_\mu \widehat{\mathcal{L}}$ and $\nabla_\omega \widehat{\mathcal{L}}$ via **automatic differentiation** (PyTorch autograd)
5. Update $\mu, \omega$ with an adaptive step-size (we use Adam, which is analogous to the paper's scheme in Eq. 10–11)

### 3.3 Generic Mean-Field ADVI Implementation

The function below is **model-agnostic**: it takes any `log_joint_fn` (a callable that maps an unconstrained vector $\zeta$ to the scalar $\log p(x, T^{-1}(\zeta)) + \log|\det J|$) and returns the optimized variational parameters.

We rely on **PyTorch's autograd** for the "AD" in ADVI — the gradients $\nabla_\mu \mathcal{L}$ and $\nabla_\omega \mathcal{L}$ are computed automatically via backpropagation through the reparameterized ELBO.

In [ ]:
LOG2PI = np.log(2.0 * np.pi)


def advi_meanfield(log_joint_fn, K, n_iter=10000, lr=0.01,
                   print_every=500, grad_clip=5.0):
    """
    Mean-field ADVI (Algorithm 1 of the paper).

    Parameters
    ----------
    log_joint_fn : callable  zeta (Tensor[K]) -> scalar Tensor
        Computes  log p(x, T^{-1}(zeta)) + log|det J_{T^{-1}}(zeta)|.
        Must be differentiable w.r.t. zeta via PyTorch autograd.
    K : int
        Total number of unconstrained latent variables.
    n_iter : int
        Maximum number of SGD iterations.
    lr : float
        Adam learning rate (analogous to eta in Eq. 10).
    print_every : int
        Print a running-average ELBO every this many iterations.
    grad_clip : float or None
        Max gradient norm (stabilises early iterations).

    Returns
    -------
    mu     : Tensor[K]   — posterior means  E_q[zeta_k]
    sigma  : Tensor[K]   — posterior std devs  exp(omega_k)
    elbos  : list[float] — ELBO at every iteration
    """
    # Initialise variational parameters (Algorithm 1: mu=0, omega=0)
    mu = torch.zeros(K, requires_grad=True)
    omega = torch.zeros(K, requires_grad=True)

    optimizer = torch.optim.Adam([mu, omega], lr=lr)
    elbos = []

    for i in range(1, n_iter + 1):
        optimizer.zero_grad()

        # --- Reparameterization trick (Section 2.5) ---
        # eta ~ N(0, I);  zeta = mu + exp(omega) * eta
        eta = torch.randn(K)
        sigma_q = torch.exp(omega)
        zeta = mu + sigma_q * eta

        # --- ELBO (Eq. 5) ---
        log_p = log_joint_fn(zeta)                      # E[log p + log|J|]
        entropy = omega.sum() + 0.5 * K * (1.0 + LOG2PI)  # H[q]
        elbo = log_p + entropy

        # --- Gradient step ---
        (-elbo).backward()
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_([mu, omega], grad_clip)
        optimizer.step()

        elbos.append(elbo.item())

        if print_every and i % print_every == 0:
            window = elbos[-min(100, len(elbos)):]
            print(f"  iter {i:>6d} | ELBO = {np.mean(window):>14.1f}  "
                  f"(± {np.std(window):.1f})")

    print(f"  ✓ Done — final ELBO ≈ {np.mean(elbos[-100:]):.1f}")
    return mu.detach().clone(), torch.exp(omega.detach().clone()), elbos

### 3.4 PPCA-ARD: log-joint decomposition

The `log_joint_fn` for ADVI must return $\log p(x, T^{-1}(\zeta)) + \log|\det J_{T^{-1}}(\zeta)|$.

We work in the **unconstrained** parameterization $\zeta = (z, \text{vec}(W), \log\sigma, \log\alpha)$ and expand the log-joint term by term:

$$\underbrace{-\frac{ND}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_n\|x_n - Wz_n\|^2}_{\text{Likelihood: } x_n \sim \mathcal{N}(Wz_n,\, \sigma^2 I)}$$

$$\underbrace{- \frac{NM}{2}\log(2\pi) - \frac{1}{2}\sum_n \|z_n\|^2}_{\text{Prior: } z_n \sim \mathcal{N}(0, I)}$$

$$\underbrace{- \frac{DM}{2}\log(2\pi) - \frac{DM}{2}\log\sigma - \frac{D}{2}\sum_m\log\alpha_m - \frac{1}{2\sigma}\sum_{d,m}\frac{w_{dm}^2}{\alpha_m}}_{\text{Prior: } w_d \sim \mathcal{N}(0,\, \sigma\,\text{diag}(\alpha))}$$

$$\underbrace{- \log\sigma - \frac{1}{2}\log(2\pi) - \frac{(\log\sigma)^2}{2}}_{\text{Prior: } \sigma \sim \text{LogNormal}(0,1)}$$

$$\underbrace{\sum_m \left[-2\log\alpha_m - \frac{1}{\alpha_m}\right]}_{\text{Prior: } \alpha_m \sim \text{InvGamma}(1,1)}$$

$$\underbrace{\log\sigma + \sum_m \log\alpha_m}_{\text{Jacobian: } \frac{d\sigma}{d\zeta_\sigma} = \sigma,\;\; \frac{d\alpha_m}{d\zeta_{\alpha_m}} = \alpha_m}$$

The vector $\zeta$ is packed as: $[\underbrace{z}_{ M \times N}\; \underbrace{\text{vec}(W)}_{D \times M}\; \underbrace{\log\sigma}_{1}\; \underbrace{\log\alpha}_{M}]$, giving $K = MN + DM + 1 + M$ total unconstrained parameters.

In [ ]:
def make_ppca_ard_log_joint(X_np, M):
    """
    Build the log-joint function for PPCA with ARD.

    Parameters
    ----------
    X_np : ndarray (N, D) — centred data matrix
    M    : int             — maximum number of latent dimensions

    Returns
    -------
    log_joint_fn : callable(zeta: Tensor[K]) -> scalar Tensor
    K            : int — total number of unconstrained latent variables
    """
    N, D = X_np.shape
    X_t = torch.tensor(X_np, dtype=torch.float64).T          # (D, N)

    z_size = M * N
    W_size = D * M
    K = z_size + W_size + 1 + M

    def log_joint(zeta):
        # ---- Unpack unconstrained vector ----
        z         = zeta[:z_size].reshape(M, N)                # (M, N)
        W         = zeta[z_size:z_size + W_size].reshape(D, M) # (D, M)
        log_sigma = zeta[z_size + W_size]                      # scalar
        log_alpha = zeta[z_size + W_size + 1:]                 # (M,)

        sigma = torch.exp(log_sigma)
        alpha = torch.exp(log_alpha)

        # ---- Likelihood:  x_n ~ N(W z_n, sigma^2 I) ----
        recon = W @ z                                          # (D, N)
        residuals = X_t - recon
        ll = (-0.5 * N * D * LOG2PI
              - N * D * log_sigma
              - 0.5 / sigma**2 * (residuals * residuals).sum())

        # ---- Prior on z:  z_n ~ N(0, I) ----
        lp_z = -0.5 * N * M * LOG2PI - 0.5 * (z * z).sum()

        # ---- Prior on W:  w_d ~ N(0, sigma * diag(alpha)) ----
        #   sum_d log N(w_d; 0, sigma*diag(alpha))
        lp_W = (-0.5 * D * M * LOG2PI
                - 0.5 * D * M * log_sigma
                - 0.5 * D * log_alpha.sum()
                - 0.5 / sigma * (W * W / alpha).sum())

        # ---- Prior on sigma:  LogNormal(0, 1) ----
        #   log f(sigma) = -log(sigma) - 0.5*log(2pi) - 0.5*(log sigma)^2
        lp_sigma = -log_sigma - 0.5 * LOG2PI - 0.5 * log_sigma**2

        # ---- Prior on alpha:  InvGamma(1, 1) ----
        #   log f(alpha_m) = -2*log(alpha_m) - 1/alpha_m
        lp_alpha = (-2.0 * log_alpha - 1.0 / alpha).sum()

        # ---- Jacobian of transform (log-scale -> positive) ----
        log_jac = log_sigma + log_alpha.sum()

        return ll + lp_z + lp_W + lp_sigma + lp_alpha + log_jac

    return log_joint, K

### 3.5 Run ADVI on a subsample

Following the paper: *"We randomly subsample ten thousand trajectories and use ADVI to infer a subspace."*

We centre the data (subtract the mean) since PPCA assumes zero-mean observations, and set $M = D = 100$ latent dimensions — the ARD prior will prune the unnecessary ones.

In [ ]:
# ---- Configuration ----
N_SUB = 10_000   # subsample size (paper: 10 000)
M_DIM = 100      # max latent dimensions (= D, paper: 100)
N_ITER = 5_000   # ADVI iterations
LR = 0.01        # Adam learning rate

# ---- Subsample and centre ----
sub_idx = np.random.choice(len(X), size=N_SUB, replace=False)
X_sub = X[sub_idx]

X_mean = X_sub.mean(axis=0)
X_sub_c = X_sub - X_mean   # centred

D = X_sub_c.shape[1]
print(f"Subsample: {N_SUB} trajectories, D = {D}")
print(f"Max latent dimensions M = {M_DIM}")

# ---- Build model ----
log_joint_fn, K_total = make_ppca_ard_log_joint(X_sub_c, M_DIM)
print(f"Total unconstrained latent variables K = {K_total:,}")
print(f"Variational parameters (mu + omega):    {2 * K_total:,}")

# ---- Run ADVI ----
print(f"\nRunning mean-field ADVI for {N_ITER} iterations (lr={LR}) ...")
mu_ppca, sigma_ppca, elbos_ppca = advi_meanfield(
    log_joint_fn, K_total,
    n_iter=N_ITER, lr=LR, print_every=500, grad_clip=5.0,
)

### 3.6 ELBO convergence

The ELBO should increase and stabilise, similar to Figure 13 of the paper.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Raw ELBO (noisy single-sample estimates)
axes[0].plot(elbos_ppca, linewidth=0.3, alpha=0.5)
window = 50
smoothed = np.convolve(elbos_ppca, np.ones(window)/window, mode="valid")
axes[0].plot(np.arange(window-1, len(elbos_ppca)), smoothed, "r-", linewidth=1.5,
             label=f"running mean (w={window})")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("ELBO")
axes[0].set_title("ELBO convergence (PPCA-ARD)")
axes[0].legend()

# Zoom on last half
half = len(elbos_ppca) // 2
axes[1].plot(range(half, len(elbos_ppca)), elbos_ppca[half:], linewidth=0.3, alpha=0.5)
axes[1].plot(np.arange(max(window-1, half), len(elbos_ppca)),
             smoothed[max(0, half-window+1):], "r-", linewidth=1.5)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("ELBO")
axes[1].set_title("ELBO convergence — last half")

plt.tight_layout()
plt.show()

### 3.7 ARD analysis — how many dimensions survived?

We extract the posterior mean of $\alpha_m$ (the ARD scale parameters). Components with **large** $\hat{\alpha}_m$ correspond to active latent dimensions; components with **small** $\hat{\alpha}_m$ have been pruned by the hierarchical prior.

We sort by $\hat{\alpha}_m$ and look for a clear gap to determine the effective dimensionality. The paper finds **11 active dimensions**.

In [ ]:
def unpack_ppca_ard(mu, N, D, M):
    """Extract posterior means from the ADVI variational mean vector."""
    z_size = M * N
    W_size = D * M
    z_hat = mu[:z_size].reshape(M, N).numpy()
    W_hat = mu[z_size:z_size + W_size].reshape(D, M).numpy()
    log_sigma_hat = mu[z_size + W_size].item()
    log_alpha_hat = mu[z_size + W_size + 1:].numpy()
    return z_hat, W_hat, np.exp(log_sigma_hat), np.exp(log_alpha_hat)

z_hat, W_hat, sigma_hat, alpha_hat = unpack_ppca_ard(
    mu_ppca, N_SUB, D, M_DIM
)

print(f"Posterior mean of sigma = {sigma_hat:.4f}")
print(f"alpha range: [{alpha_hat.min():.4e}, {alpha_hat.max():.4e}]")

# Sort alpha in descending order
order = np.argsort(alpha_hat)[::-1]
alpha_sorted = alpha_hat[order]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(M_DIM), alpha_sorted, color="steelblue", edgecolor="none")
axes[0].set_xlabel("Component (sorted)")
axes[0].set_ylabel(r"$\hat{\alpha}_m$")
axes[0].set_title("ARD weights — all components")

axes[1].bar(range(min(30, M_DIM)), alpha_sorted[:30], color="steelblue", edgecolor="none")
axes[1].set_xlabel("Component (sorted)")
axes[1].set_ylabel(r"$\hat{\alpha}_m$")
axes[1].set_title("ARD weights — top 30 (zoom)")

plt.tight_layout()
plt.show()

# Determine effective dimensionality via the "elbow" in sorted alpha
log_alpha_sorted = np.log10(alpha_sorted + 1e-30)
gaps = np.diff(log_alpha_sorted)
elbow = np.argmin(gaps) + 1  # first big drop
print(f"\nEffective dimensionality (elbow heuristic): {elbow}")
print(f"Top {elbow} alpha values: {alpha_sorted[:elbow].round(4)}")

### 3.8 Project all trajectories into the learned subspace

Given the posterior mean $\hat{W}$ (only the **active** columns) and $\hat{\sigma}$, the posterior mean of each latent code is:

$$\mathbb{E}[z_n \mid x_n] = \left(\hat{W}_{\text{active}}^\top \hat{W}_{\text{active}} + \hat{\sigma}^2 I\right)^{-1} \hat{W}_{\text{active}}^\top\, (x_n - \bar{x})$$

This projects each trajectory from $\mathbb{R}^{100}$ into $\mathbb{R}^{M_{\text{active}}}$.

In [ ]:
# Keep only the active dimensions
active_dims = order[:elbow]
W_active = W_hat[:, active_dims]                          # (D, M_active)
M_active = len(active_dims)

# Posterior projection matrix:  (W'W + sigma^2 I)^{-1} W'
WtW = W_active.T @ W_active                               # (M_active, M_active)
M_mat = WtW + sigma_hat**2 * np.eye(M_active)
proj_matrix = np.linalg.solve(M_mat, W_active.T)          # (M_active, D)

# Project the FULL dataset (centred with the same mean)
X_centred = X - X_mean
Z_all = X_centred @ proj_matrix.T                          # (N_full, M_active)

print(f"Active dimensions: {M_active}")
print(f"Projection:  {X.shape} -> {Z_all.shape}")
print(f"W_active shape: {W_active.shape}")
print(f"sigma_hat = {sigma_hat:.4f}")